## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

### Set-up

In [2]:
import torch
import gc
import random
import pandas as pd
import re
from tqdm import tqdm

import sys
sys.path.append("src")
import _dataset
import _prompt
import _mapping
import _util
from _intervention import forward_with_cache, prepare_batch_token_intervention, batch_intervene, get_attention_freeze_hooks

import pyvene
from pyvene import (
    IntervenableModel,
    VanillaIntervention,
    CollectIntervention,
    BoundlessRotatedSpaceIntervention,
    RepresentationConfig,
    IntervenableConfig,
)
from pyvene import set_seed, count_parameters

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A5000
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

In [4]:
model, tokenizer = _util.load_OSS()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Two-Digit Patching (Whole Dataset)

In [5]:
LAYER = 0

# Load the divided prompts dataset
divided_prompts = pd.read_csv("data/hundreds_three_digit_sums_pre_result_divided_prompts.csv")
divided_prompts["base_number"] = divided_prompts["base_number"].astype('Int64')
divided_prompts["source_number"] = divided_prompts["source_number"].astype('Int64')
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 2560 divided prompts


In [7]:
header = list(divided_prompts.columns) + ['generated_text']
filepath = _util.create_csv_file("experiments/activation_intervention/output/GPT_OSS_stepwise", f"attention_freeze_hundreds_three_digit_sums_pre_result.csv", header)

batch_size = 16
df = divided_prompts[divided_prompts['intervention_id'] == 25]

for i in tqdm(range(0, len(df), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = df.iloc[i:i+batch_size]
    
    # Prepare batch of intervention prompts
    base_before = batch_rows['base_before'].tolist()
    base_number = batch_rows['base_number'].tolist()
    base_after = batch_rows['base_after'].tolist()
    source_before = batch_rows['source_before'].tolist()
    source_number = batch_rows['source_number'].tolist()
    source_after = batch_rows['source_after'].tolist()

    tokens, hook = prepare_batch_token_intervention(model, tokenizer, LAYER, base_before, base_number, base_after, source_before, source_number)
    input_length = tokens["input_ids"].shape[1]

    for j in range(3):
        attention_freeze_hooks = get_attention_freeze_hooks(model, tokens)
        with torch.no_grad():
            output = batch_intervene(model, tokens["input_ids"], attention_freeze_hooks + [hook], attention_mask=tokens["attention_mask"])
        pred_toks = output.logits[:,-1,:].argmax(dim=-1)
        tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
        del output
    
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [generated_text])

    del tokens, hook
    torch.cuda.empty_cache()
    gc.collect()


  0%|                                                                                                    | 0/16 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████| 16/16 [07:30<00:00, 28.18s/it]
